# Compute Raw PTR Values
April 2, 2024

Recompute the raw PTR values from the input G data. Update to use the 0.01 pseudocounts.


In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [34]:
from src.reference_data import load_analysis_genes
import glob 

def load_g_files(chromatin_dir):
    """Load all of the gene G results into a dataframe, 
    flatten the F images for the dataframe."""

    geneset = load_analysis_genes()
        
    # Check the F images for each gene, determine if we 
    # need to modify the ptr calculation
    # In case +1 is too high of a psuedo count, we will reduce the pseudo count

    g_filepaths = glob.glob(f'{chromatin_dir}/*_g_*.npy')
    
    # Load the F images for each deconvolved gene
    current_g = np.load(g_filepaths[0])
    n_times, y_bins, x_bins = current_g.shape

    all_gene_gs_df = pd.DataFrame(index=geneset.index, 
       columns=np.arange(n_times*y_bins*x_bins))

    from src.timer import Timer

    timer = Timer()
    i = 0

    # For each deconvolved gene, load the ptr values and 
    # place them into the PTRs dataframe
    for path in g_filepaths:
        filename = path.split('/')[-1]
        orf_name = filename.split('_')[2]

        # Skip genes not in our analysis set
        # for runs in which we haven't filtered for low coverage genes yet
        if not orf_name in geneset.index.values: continue

        current_g = np.load(path)
        
        try:
            all_gene_gs_df.loc[orf_name] = current_g.flatten()
        except ValueError:
            print(f"Error with orf: {orf_name}. Skipping")
            i += 1
            continue

        if i % 1000 == 0:
            timer.print_time(f"{i+1}/{len(g_filepaths)}")
        i += 1

    return all_gene_gs_df


In [35]:
# We did save the G's in replicate 1, so let's load the G values for now 
# from these directories

rep1_chrom_dir = 'output/archive/deconvolve_rep1_g006_2024_02_20/chromatin/'
rep2_chrom_dir = 'output/archive/deconvolve_rep2_g006_2024_02_22/chromatin/'


In [38]:
rep2_gs_df = load_g_files(rep2_chrom_dir)

1/4739 - 00:00:00.003
1001/4739 - 00:00:01.758
2001/4739 - 00:00:03.436
3001/4739 - 00:00:05.074
4001/4739 - 00:00:06.801


In [75]:
rep2_gs_df = rep2_gs_df.dropna()

In [76]:
rep2_gs_df.shape

(4677, 12000)

In [82]:
rep2_g_imgs = rep2_gs_df.values.reshape((-1, 15, 16, 50))
rep2_g_imgs.shape

(4677, 15, 16, 50)

In [ ]:
from src.config import load_yl_rg1_vst_config, get_yl2019_chromatin_timepoints
from src.chromatin_model import ChromatinModel

cur_g = rep2_g_imgs
replicate = 2
config = load_yl_rg1_vst_config(replicate)
timepoints = get_yl2019_chromatin_timepoints(replicate)

# Compute the raw PTR for the gene for both replicates
from src.peak_to_trough import compute_quantile_ptr

mu0 = config.intervals_wt1[0][0]

# Compute the peak to trough ratio for non recovery timepoints
non_RG1_G = cur_g[:, timepoints > mu0, :]
gene_g_ptrs = np.apply_along_axis(lambda mat: compute_quantile_ptr(mat, 
    0.2, 0.8, eps=1), 1, non_RG1_G)

In [ ]:
mean_g_ptrs = gene_g_ptrs.mean(axis=1).mean(axis=1)

In [ ]:
plt.hist(mean_g_ptrs.flatten(), bins=100)
0